# Experiment 6: Bagging, Boosting, and Stacked Ensemble Models
This standalone notebook implements Bagging, Boosting (AdaBoost, Gradient Boosting), and a Stacking Ensemble on the Wisconsin Diagnostic Breast Cancer dataset.

In [1]:
import os
import matplotlib
matplotlib.use('Agg') # Strictly headless - non-interfering, zero GUI popups

def resolve_path(rel_path):
    """Dynamically resolves datasets whether run from repo root or Ex subfolder."""
    for prefix in ['', '../', '../../']:
        cand = os.path.join(prefix, rel_path)
        if os.path.exists(cand):
            return cand
    return rel_path

def resolve_out(rel_path):
    """Avoids nested directories if running from within Ex6."""
    if os.path.basename(os.getcwd()) == 'Ex6':
        if rel_path.startswith('Ex6/'):
            return rel_path[len('Ex6/'):]
    return rel_path

import pandas as pd
import numpy as np
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs(resolve_out('Ex6'), exist_ok=True)

In [2]:
def run_experiment_6(csv_path="Datasets/Breast_Cancer/breast_cancer.csv"):
    print("="*60)
    print("=== LAUNCHING EXPERIMENT 6: ENSEMBLE MODELS ===")
    print("="*60)
    
    path = resolve_path(csv_path)
    df = pd.read_csv(path)
    if 'id' in df.columns:
        df = df.drop(columns=['id'])
    if 'Unnamed: 32' in df.columns:
        df = df.drop(columns=['Unnamed: 32'])
        
    target_col = 'diagnosis' if 'diagnosis' in df.columns else df.columns[0]
    X = df.drop(columns=[target_col])
    y = df[target_col].map({'M': 1, 'B': 0}) if df[target_col].dtype == 'object' else df[target_col]
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
    
    base_dt = DecisionTreeClassifier(max_depth=4, random_state=42)
    models = {
        'Bagging': BaggingClassifier(estimator=base_dt, n_estimators=50, random_state=42),
        'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, random_state=42),
        'Stacking': StackingClassifier(
            estimators=[('dt', base_dt), ('svm', SVC(probability=True, random_state=42))],
            final_estimator=LogisticRegression(), cv=5
        )
    }
    
    results = {}
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1]
        cv_acc = np.mean(cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy'))
        results[name] = {
            'Accuracy (%)': round(accuracy_score(y_te, y_pred) * 100, 2),
            'CV Accuracy (%)': round(cv_acc * 100, 2),
            'Precision (%)': round(precision_score(y_te, y_pred, zero_division=0) * 100, 2),
            'Recall (%)': round(recall_score(y_te, y_pred, zero_division=0) * 100, 2),
            'F1-Score (%)': round(f1_score(y_te, y_pred, zero_division=0) * 100, 2),
            'ROC AUC': round(roc_auc_score(y_te, y_prob), 4)
        }
        
    print("\n=== EXPERIMENT 6 PIPELINE COMPLETE ===")
    return results

In [3]:
# Master Execution Cell
ex6_output = run_experiment_6()
display(pd.DataFrame(ex6_output).T.style.background_gradient(cmap='Blues', subset=['Accuracy (%)', 'F1-Score (%)']))

=== LAUNCHING EXPERIMENT 6: ENSEMBLE MODELS ===



=== EXPERIMENT 6 PIPELINE COMPLETE ===


,Accuracy (%),CV Accuracy (%),Precision (%),Recall (%),F1-Score (%),ROC AUC
Bagging,93.860000,95.080000,94.520000,95.830000,95.170000,0.990700
AdaBoost,95.610000,96.840000,94.670000,98.610000,96.600000,0.982500
Gradient Boosting,94.740000,95.960000,94.590000,97.220000,95.890000,0.988400
Stacking,96.490000,97.010000,97.220000,97.220000,97.220000,0.994400
